## 1. RetinaNet

**Contents:**
1. What & Why
2. Architecture
3. The Class-Imbalance Problem
4. Focal Loss
5. torchvision Usage
6. References

---

**RetinaNet** (Lin, Goyal, Girshick, He, Dollar, ICCV 2017 — "Focal Loss for Dense Object Detection") is a **one-stage** detector that, for the first time, matched the **accuracy** of two-stage detectors like Faster R-CNN while keeping the **speed** of one-stage detectors.

The paper's central claim is that one-stage detectors had historically been *less accurate* not because of their architecture, but because of a severe **foreground-background class imbalance** during training. RetinaNet's main contribution is a new loss — **Focal Loss** — that fixes this, plus a clean architecture built on a **Feature Pyramid Network (FPN)**.

## 2. Architecture

```
Image
  |
  v
Backbone (ResNet-50) + FPN  ->  pyramid levels P3..P7
  |
  +-- at every level, two small shared FCN sub-nets (applied densely):
  |
  +--> Classification subnet:  K object classes x A anchors  (sigmoid per class)
  +--> Box-regression subnet:  4 box deltas x A anchors
  |
  v
Decode anchors + per-class NMS  ->  detections
```

Key points:

- **One stage:** unlike Faster R-CNN there is **no RPN and no proposal step**. The two subnets predict class and box **directly and densely** on every anchor at every FPN location.
- **FPN** gives a multi-scale feature pyramid (P3..P7) so each level handles a range of object sizes.
- **Anchors:** at each location there are `A` anchors (typically 9 = 3 scales x 3 aspect ratios). This dense tiling produces ~100k anchors per image — the source of the imbalance below.
- The classification subnet uses an independent **sigmoid per class** (not a softmax), so detection is treated as `K` one-vs-rest binary problems.

## 3. The Class-Imbalance Problem

A dense one-stage detector evaluates ~10^4–10^5 anchor locations per image, but only a handful contain objects. So the training set per image is overwhelmingly **easy background (negative)** examples.

With standard cross-entropy, each easy negative contributes a small but non-zero loss; summed over 100k of them, the **easy negatives dominate the gradient** and drown out the rare, informative foreground examples. Two-stage detectors avoid this because the RPN first filters down to ~1–2k proposals and then uses sampling with a fixed fg:bg ratio. One-stage detectors had relied on heuristics like hard-negative mining; RetinaNet instead changes the loss itself.

## 4. Focal Loss

**Focal Loss** reshapes standard cross-entropy so that **well-classified (easy) examples are down-weighted**, letting the rare hard examples drive learning.

Starting from cross-entropy for a binary case, define `p_t` as the model's probability of the *true* class:

```
CE(p_t) = -log(p_t)
```

Focal Loss multiplies by a **modulating factor** `(1 - p_t)^gamma` and an optional balancing weight `alpha_t`:

```
FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
```

- When an example is **easy** (`p_t -> 1`), `(1 - p_t)^gamma -> 0`, so its loss is suppressed.
- When an example is **hard** (`p_t` small), the factor is near 1, so it behaves like normal cross-entropy.
- **gamma** (the *focusing* parameter) controls how aggressively easy examples are down-weighted; `gamma = 0` recovers cross-entropy. The paper uses **gamma = 2, alpha = 0.25**.

This single change is what lets a one-stage detector train on all anchors densely without being swamped by background.

## 5. torchvision Usage

```python
import torch
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights,
)

weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT
model = retinanet_resnet50_fpn(weights=weights)
model.eval()
preprocess = weights.transforms()

from torchvision.io import read_image
img = read_image("image.jpg")           # uint8 CxHxW
batch = [preprocess(img)]

with torch.no_grad():
    out = model(batch)[0]

print(out["boxes"])   # (N, 4)  [x1, y1, x2, y2]
print(out["labels"])  # (N,)
print(out["scores"])  # (N,)

categories = weights.meta["categories"]
keep = out["scores"] > 0.5
for box, label in zip(out["boxes"][keep], out["labels"][keep]):
    print(categories[label], box.tolist())
```

During training, pass images and targets (`boxes` + `labels`) and the model returns a loss dict with the **classification (focal) loss** and the **box-regression loss**:

```python
model.train()
images  = [torch.rand(3, 600, 800)]
targets = [{"boxes": torch.tensor([[10., 20., 200., 300.]]),
            "labels": torch.tensor([1])}]
loss_dict = model(images, targets)   # {'classification', 'bbox_regression'}
loss = sum(loss_dict.values())
loss.backward()
```

torchvision also exposes the focal loss directly as `torchvision.ops.sigmoid_focal_loss`, and an improved model `retinanet_resnet50_fpn_v2`.

## 6. References

- Lin, Goyal, Girshick, He, Dollar, *Focal Loss for Dense Object Detection*, ICCV 2017 — https://arxiv.org/abs/1708.02002
- Lin et al., *Feature Pyramid Networks for Object Detection*, CVPR 2017 — https://arxiv.org/abs/1612.03144
- torchvision detection models — https://pytorch.org/vision/stable/models.html#object-detection
- `torchvision.ops.sigmoid_focal_loss` — https://pytorch.org/vision/stable/generated/torchvision.ops.sigmoid_focal_loss.html